# Generating Batch Inputs

pytauargus can also **write** the batch input files, as a port of
[rtauargus](https://github.com/InseeFrLab/rtauargus) (the R wrapper).
The generators emit exactly the file formats the engine reads, so
you can script the full *input → batch → run* flow in pure Python.

| Function | Output | Ported from (R) |
|----------|--------|-----------------|
| `micro_arb(…)` | `.arb` batch file | `micro_arb.R::micro_arb` |
| `write_rda` / `rda_text` | `.rda` metadata text | `micro_asc_rda.R::write_rda` |
| `write_hrc(…)` | `.hrc` hierarchy file | `hrc.R::write_hrc` |

> **Gap:** the fixed-width `.asc` microdata writer (`gdata::write.fwf`)
> is **not** ported. Produce your own `.asc` (pandas, R, …); the
> generators only *reference* it by name.

## Setup

In [1]:
from pathlib import Path
import pytauargus

# Locate the repo's data/ fixtures (works from notebooks/ or repo root)
_here = Path.cwd()
DATA = next(
    (p / "data") for p in [_here, *_here.parents]
    if (p / "data" / "tau_testW.asc").exists()
)
OUT = _here / "out"          # scratch dir for notebook outputs (gitignored)
OUT.mkdir(exist_ok=True)
print(f"pytauargus {pytauargus.__version__}")
print(f"data : {DATA}")
print(f"out  : {OUT}")

pytauargus 0.2.0
data : /Users/trygve/local-code/tauargus/data
out  : /Users/trygve/local-code/tauargus/bindings/python/out


## 1. `micro_arb` — write a `.arb` batch file

The main generator. Signature:

```python
micro_arb(
    arb_filename: str,          # output .arb path (default: temp file)
    asc_filename: str,          # input .asc (referenced, not created)
    rda_filename: str | None,  # defaults to asc_filename with .rda
    explanatory_vars: list,    # ["A","B"] = 1 table; [["A","B"],["C"]] = 2
    response_var: str | list,  # response variable name(s)
    safety_rules: str | list,  # e.g. "NK(1,85)"; recycled if shorter
    suppress: str | list,      # e.g. "OPT(.,0)" — "." = table number
    output_names: list,        # one per table
) -> {"arb_filename": str, "output_names": list}
```

In [2]:
from pytauargus.arb import micro_arb

res = micro_arb(
    arb_filename=str(OUT / "demo.arb"),
    asc_filename=str(DATA / "tau_testW.asc"),
    rda_filename=str(DATA / "tau_testW.rda"),
    explanatory_vars=[
        ["Size", "Region"],         # table 1
        ["Region", "IndustryCode"], # table 2
    ],
    response_var="Var2",
    safety_rules=["NK(2,75)"],     # recycled across both tables
    suppress="OPT(.,0)",           # "." -> table number; 0 = no time limit
    output_names=["out1.csv", "out2.csv"],
)
print(res)
print()
print("=== generated demo.arb ===")
print((OUT / "demo.arb").read_text())

{'arb_filename': '/Users/trygve/local-code/tauargus/bindings/python/out/demo.arb', 'output_names': ['out1.csv', 'out2.csv']}

=== generated demo.arb ===
// Batch generated by package *rtauargus*
// (2026-10-10 12:14:09 CEST)
<OPENMICRODATA> "/Users/trygve/local-code/tauargus/data/tau_testW.asc"
<OPENMETADATA> "/Users/trygve/local-code/tauargus/data/tau_testW.rda"
<SPECIFYTABLE> "Size""Region"|"Var2"||
<SAFETYRULE> NK(2,75)
<SPECIFYTABLE> "Region""IndustryCode"|"Var2"||
<SAFETYRULE> NK(2,75)
<READMICRODATA>
<SUPPRESS> OPT(1,0)
<WRITETABLE> (1,4,,"out1.csv")
<SUPPRESS> OPT(2,0)
<WRITETABLE> (2,4,,"out2.csv")




### Per-table parameters

`safety_rules`, `suppress`, `output_type`, … all accept a single
value (recycled) or a per-table list:

In [3]:
res2 = micro_arb(
    arb_filename=str(OUT / "demo2.arb"),
    asc_filename=str(DATA / "tau_testW.asc"),
    rda_filename=str(DATA / "tau_testW.rda"),
    explanatory_vars=[["Size"], ["Region"]],
    response_var="Var2",
    safety_rules=["NK(1,85)", "FREQ(3,10)"],   # different rule per table
    suppress="MOD(.,0)",
    output_names=["s1.csv", "s2.csv"],
)
print((OUT / "demo2.arb").read_text())

// Batch generated by package *rtauargus*
// (2026-10-10 12:14:09 CEST)
<OPENMICRODATA> "/Users/trygve/local-code/tauargus/data/tau_testW.asc"
<OPENMETADATA> "/Users/trygve/local-code/tauargus/data/tau_testW.rda"
<SPECIFYTABLE> "Size"|"Var2"||
<SAFETYRULE> NK(1,85)
<SPECIFYTABLE> "Region"|"Var2"||
<SAFETYRULE> FREQ(3,10)
<READMICRODATA>
<SUPPRESS> MOD(1,0)
<WRITETABLE> (1,4,,"s1.csv")
<SUPPRESS> MOD(2,0)
<WRITETABLE> (2,4,,"s2.csv")




## 2. `write_rda` / `rda_text` — write `.rda` metadata

Each variable is a dict describing its position, width, type, and
optional codelist / hierarchy:

| Key | Required | Meaning |
|-----|----------|---------|
| `colname` | yes | Variable name |
| `position` | yes | 1-based byte position |
| `width` | yes | Field width |
| `type_var` | yes | `NUMERIC`, `RECODEABLE`, `WEIGHT`, `HOLDING` |
| `digits` | yes* | Decimal places (*NUMERIC / WEIGHT only) |
| `missing` | no | Missing-value code (``/`None` = none) |
| `totcode` | no | Total-code string |
| `codelist` | no | `.cdl` path (normalised to absolute) |
| `hierarchical` | no | `.hrc` file, level string like `2 1`, or `None` |
| `hierleadstring` | no | Hierarchy lead string (`.hrc` files) |

`write_rda` returns one block string per variable; `rda_text`
renders the whole file.

In [4]:
from pytauargus.rda import write_rda, rda_text

info_vars = [
    dict(type_var="RECODEABLE", colname="V1", position=1, width=1,
         digits=0, missing="?", totcode="Total", codelist=None,
         hierarchical=None, hierleadstring=None),
    dict(type_var="RECODEABLE", colname="V2", position=3, width=1,
         digits=0, missing="", totcode="Total", codelist=None,
         hierarchical="V2.hrc", hierleadstring="@"),
    dict(type_var="NUMERIC", colname="VAL", position=5, width=3,
         digits=2, missing="9999", totcode=None, codelist=None,
         hierarchical=None, hierleadstring=None),
    dict(type_var="WEIGHT", colname="POIDS", position=9, width=4,
         digits=1, missing="#", totcode=None, codelist=None,
         hierarchical=None, hierleadstring=None),
]

blocks = write_rda(info_vars)
print(f"{len(blocks)} variable blocks")
for b in blocks:
    print(repr(b))

print()
print("=== full .rda text ===")
print(rda_text(info_vars))

4 variable blocks
'V1 1 1 ?\n  <RECODEABLE>\n  <TOTCODE> "Total"'
'V2 3 1\n  <RECODEABLE>\n  <TOTCODE> "Total"\n  <HIERARCHICAL>\n  <HIERCODELIST> "V2.hrc"\n  <HIERLEADSTRING> "@"'
'VAL 5 3 9999\n  <NUMERIC>\n  <DECIMALS> 2'
'POIDS 9 4 #\n  <WEIGHT>\n  <DECIMALS> 1'

=== full .rda text ===
V1 1 1 ?
  <RECODEABLE>
  <TOTCODE> "Total"
V2 3 1
  <RECODEABLE>
  <TOTCODE> "Total"
  <HIERARCHICAL>
  <HIERCODELIST> "V2.hrc"
  <HIERLEADSTRING> "@"
VAL 5 3 9999
  <NUMERIC>
  <DECIMALS> 2
POIDS 9 4 #
  <WEIGHT>
  <DECIMALS> 1



## 3. `write_hrc` — write a `.hrc` hierarchy file

Builds a hierarchy from microdata columns ordered **finest →
coarsest** and writes it as a `.hrc` file (child levels are
prefixed with the lead string, default `@`).

```python
write_hrc(
    microdata: dict,      # {var: [value, …]}  (values: str | None)
    vars_hrc: str | list, # column name(s), finest → coarsest
    hierleadstring: str = "@",
    hrc_filename: str | None,   # default: temp file
    fill_na: str = "up",        # "up"/"down" NA imputation
    compact: bool = True,       # prune single-value branches
    hierlevels: str | None,     # split one code into positional levels
) -> str  # the written path
```

In [5]:
from pytauargus.hrc import write_hrc
import warnings

# A small hierarchy: city (finest) → country → continent
microdata = {
    "city":      ["PAR", "LYO", "BER", "MUC", "NYC", "TOR", "TYO"],
    "country":   ["FR",  "FR",  "DE",  "DE",  "US",  "CA",  "JP"],
    "continent": ["EU",  "EU",  "EU",  "EU",  "NA",  "NA",  "AS"],
}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # silence single-level/NA warnings in demos
    hrc_path = write_hrc(
        microdata,
        vars_hrc=["city", "country", "continent"],
        hrc_filename=str(OUT / "geo.hrc"),
    )

print(f"wrote {hrc_path}")
print()
print((OUT / "geo.hrc").read_text())

wrote /Users/trygve/local-code/tauargus/bindings/python/out/geo.hrc

AS
@JP
@@TYO
EU
@DE
@@BER
@@MUC
@FR
@@LYO
@@PAR
NA
@CA
@@TOR
@US
@@NYC



### `hierlevels` — split one code into positional levels

When a single column encodes several levels positionally (e.g.
`"123"` = level 1 `1`, level 2 `2`, level 3 `3`), `hierlevels`
splits it — exactly one `vars_hrc` column is allowed:

In [6]:
import warnings

microdata2 = {"code": ["111", "112", "121", "122", "211", "212"]}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    hrc2 = write_hrc(
        microdata2,
        vars_hrc="code",
        hierlevels="1 1 1",           # 3 levels, 1 character each
        hrc_filename=str(OUT / "pos.hrc"),
    )

print((OUT / "pos.hrc").read_text())

1
@11
@@111
@@112
@12
@@121
@@122
2
@21
@@211
@@212



## 4. Round-trip: generate → run → inspect

A generated `.arb` feeds straight back into the engine. This is
the full scripted pipeline — note the in-arb `<SUPPRESS> OPT(…)`
runs during `run_batch`:

In [7]:
from pytauargus.engine import run_batch

arb_path = OUT / "demo.arb"
print(f"Running {arb_path.name}…\n")

eng = run_batch(arb_path)
for i in range(eng._n_tables):
    ncell, _ = eng._tau.get_total_table_size(i)
    spec, _ = eng._tables[i]
    print(f"  table {i+1}: {' x '.join(spec.exp_vars):<24} | {spec.resp_var}  →  {ncell} cells")

Running demo.arb…



  table 1: Size x Region            | Var2  →  162 cells
  table 2: Region x IndustryCode    | Var2  →  12798 cells


### Where did the outputs go?

`WRITETABLE` output files are written relative to the **`.arb`
file's directory** — here, the `out/` scratch dir:

In [8]:
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:<16} {p.stat().st_size:>8} bytes")

  demo.arb              462 bytes
  demo2.arb             438 bytes
  geo.hrc                71 bytes
  out1.csv             3338 bytes
  out2.csv           276737 bytes
  pos.hrc                52 bytes
  table1.csv            700 bytes
  table1.tab          10976 bytes


## Gotchas & known limitations

- `suppress="OPT(.,0)"` uses max-time `0` (no limit), so this example
  can finish without a deadline. Positive time limits are in minutes.
- The `.asc` microdata writer is **not** ported (R's
  `gdata::write.fwf`); bring your own fixed-width file.
- `write_hrc` with a **single** column emits a flat sorted list and
  warns; `fill_na` warns when NAs are imputed.

## Summary

You've now:

1. Generated `.arb` batch files with `micro_arb`
2. Generated `.rda` metadata text with `write_rda` / `rda_text`
3. Generated `.hrc` hierarchy files with `write_hrc`
4. Round-tripped: generated batch → engine → tables + CSVs

**Back to:** [01_quickstart.ipynb](01_quickstart.ipynb) ·
[02_protection_and_audit.ipynb](02_protection_and_audit.ipynb)